# Ranked Multi-Factor Rotation Equity Curve

This notebook plots the Ranked Multi-Factor Rotation strategy's equity curve over the maximum honest full-11-ticker-universe window (IGOV's real 2009-01-30 inception through the most recent fully-closed month at acquisition time). It reads the cached monthly curve from `outputs/ranked_multi_factor_rotation/ranked_multi_factor_rotation_equity_curve.csv` when available; set `REFRESH_BACKTEST = True` below to recompute it from the local raw acquired data. See `research/strategies/ranked_multi_factor_rotation/docs/reproducibility_findings.md` for full provenance -- these are raw, unvalidated backtest results (no walk-forward validation has been run), not a signal-quality claim.

In [ ]:
from pathlib import Path
import sys


def resolve_project_root() -> Path:
    candidates = []
    try:
        candidates.append(Path.cwd())
    except FileNotFoundError:
        pass
    candidates.append(Path("/Users/gavinhussey/Downloads/AtlasQuant"))
    for candidate in candidates:
        if candidate.name == "ranked_multi_factor_rotation":
            candidate = candidate.parent
        if (candidate / "src" / "atlas_quant").exists():
            return candidate
    raise FileNotFoundError("Could not locate AtlasQuant project root")


PROJECT_ROOT = resolve_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

OUTPUT_DIR = PROJECT_ROOT / "outputs" / "ranked_multi_factor_rotation"
CURVE_CSV = OUTPUT_DIR / "ranked_multi_factor_rotation_equity_curve.csv"
RAW_ROOT = PROJECT_ROOT / "data" / "raw" / "ranked_multi_factor_rotation"

START_MONTH_END = "2009-01-30"
END_MONTH_END = "2026-06-30"
REFRESH_BACKTEST = False

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("project root:", PROJECT_ROOT)
print("curve cache:", CURVE_CSV)


In [ ]:
import pandas as pd
from datetime import date


def run_backtest_equity_curve() -> pd.DataFrame:
    from atlas_quant.strategies.ranked_multi_factor_rotation.acquisition.run_acquisition import (
        load_raw_observations,
    )
    from atlas_quant.backtest.ranked_multi_factor_rotation_runner import (
        RmfrBacktestConfig,
        RmfrBacktestDependencies,
        build_dependencies_from_observations,
        run_ranked_multi_factor_rotation_backtest,
    )
    from atlas_quant.strategies.ranked_multi_factor_rotation.backtest_clock import generate_monthly_periods
    from atlas_quant.strategies.ranked_multi_factor_rotation.config import RankedMultiFactorRotationConfig
    from atlas_quant.data.point_in_time import ListTradingCalendar

    observations = load_raw_observations(RAW_ROOT)
    price_frames, close_price_source = build_dependencies_from_observations(observations)
    all_dates = sorted({o.trading_date for o in observations})
    calendar = ListTradingCalendar(trading_days=tuple(all_dates))
    dependencies = RmfrBacktestDependencies(
        ohlc_price_frames=price_frames, close_price_source=close_price_source, trading_calendar=calendar,
    )
    config = RmfrBacktestConfig(strategy_config=RankedMultiFactorRotationConfig())

    periods = generate_monthly_periods(
        date.fromisoformat(START_MONTH_END), date.fromisoformat(END_MONTH_END), calendar,
    )
    result = run_ranked_multi_factor_rotation_backtest(periods, dependencies, config)

    rows = [
        {"month_end": month_end.isoformat(), "ranked_multi_factor_rotation_equity": equity}
        for month_end, equity in result.equity_curve()
    ]
    df = pd.DataFrame(rows)
    df.to_csv(CURVE_CSV, index=False)
    return df


if CURVE_CSV.exists() and not REFRESH_BACKTEST:
    curve = pd.read_csv(CURVE_CSV)
else:
    curve = run_backtest_equity_curve()

curve["month_end"] = pd.to_datetime(curve["month_end"])
curve = curve.sort_values("month_end").reset_index(drop=True)
print(f"month-end observations: {len(curve)}")
curve.tail()


In [ ]:
from html import escape
from IPython.display import HTML, display


def render_equity_curve_svg(df, *, width=980, height=540):
    if df.empty:
        return "<p>No equity-curve data available.</p>"

    margin = {"top": 46, "right": 34, "bottom": 58, "left": 72}
    plot_w = width - margin["left"] - margin["right"]
    plot_h = height - margin["top"] - margin["bottom"]
    values = list(df["ranked_multi_factor_rotation_equity"])
    y_min = min(1.0, min(values))
    y_max = max(1.0, max(values))
    padding = (y_max - y_min) * 0.08 or 0.1
    y_min -= padding
    y_max += padding

    start = df["month_end"].min()
    end = df["month_end"].max()
    span_days = max((end - start).days, 1)

    def x_at_date(dt):
        return margin["left"] + ((dt - start).days / span_days) * plot_w

    def y_at(v):
        return margin["top"] + (y_max - v) / (y_max - y_min) * plot_h

    polyline = " ".join(
        f"{x_at_date(row.month_end):.1f},{y_at(row.ranked_multi_factor_rotation_equity):.1f}"
        for row in df.itertuples(index=False)
    )

    grid = []
    labels = []
    for j in range(6):
        value = y_min + (y_max - y_min) * j / 5
        y = y_at(value)
        grid.append(f'<line x1="{margin["left"]}" y1="{y:.1f}" x2="{width - margin["right"]}" y2="{y:.1f}" stroke="#e5e7eb"/>')
        labels.append(f'<text x="{margin["left"] - 12}" y="{y + 4:.1f}" text-anchor="end" font-size="12" fill="#4b5563">{value:.2f}x</text>')

    year_ticks = pd.date_range(start=start, end=end, freq="YS")
    if len(year_ticks) == 0 or year_ticks[0] != start:
        year_ticks = year_ticks.union(pd.DatetimeIndex([start]))
    if year_ticks[-1] != end:
        year_ticks = year_ticks.union(pd.DatetimeIndex([end]))
    x_labels = []
    for tick in year_ticks:
        x = x_at_date(tick)
        label = tick.strftime("%Y")
        x_labels.append(f'<text x="{x:.1f}" y="{height - 24}" text-anchor="middle" font-size="12" fill="#4b5563">{escape(label)}</text>')

    final_equity = df["ranked_multi_factor_rotation_equity"].iloc[-1]
    svg = f'''
    <svg viewBox="0 0 {width} {height}" width="100%" height="{height}" role="img" aria-label="Ranked Multi-Factor Rotation equity curve">
      <rect width="{width}" height="{height}" fill="#ffffff"/>
      <text x="{margin['left']}" y="24" font-size="20" font-weight="700" fill="#111827">Ranked Multi-Factor Rotation Equity Curve</text>
      <text x="{margin['left']}" y="42" font-size="12" fill="#6b7280">Monthly rebalance, net of 10bps turnover-based transaction costs. Raw backtest result -- not walk-forward validated.</text>
      {"".join(grid)}
      {"".join(labels)}
      <line x1="{margin['left']}" y1="{height - margin['bottom']}" x2="{width - margin['right']}" y2="{height - margin['bottom']}" stroke="#9ca3af"/>
      <line x1="{margin['left']}" y1="{margin['top']}" x2="{margin['left']}" y2="{height - margin['bottom']}" stroke="#9ca3af"/>
      {"".join(x_labels)}
      <polyline points="{polyline}" fill="none" stroke="#2563eb" stroke-width="2.4" stroke-linejoin="round" stroke-linecap="round"/>
      <rect x="{width - 300}" y="54" width="235" height="46" rx="6" fill="#ffffff" stroke="#d1d5db"/>
      <line x1="{width - 280}" y1="77" x2="{width - 240}" y2="77" stroke="#2563eb" stroke-width="4"/>
      <text x="{width - 230}" y="81" font-size="13" fill="#111827">Ranked Multi-Factor Rotation: {final_equity:.2f}x</text>
    </svg>
    '''
    return svg


display(HTML(render_equity_curve_svg(curve)))


## Sub-window totals

5/10/15-year totals over the same run, same config -- just a shorter trailing slice of the same equity curve, not independently re-optimized windows.

In [ ]:
def sub_window_stats(df: pd.DataFrame, years: int) -> dict:
    end = df["month_end"].max()
    start_cutoff = end - pd.DateOffset(years=years)
    window = df[df["month_end"] > start_cutoff].reset_index(drop=True)
    if window.empty:
        return {"years": years, "start": None, "end": None, "total_return": None, "months": 0}
    start_equity = df[df["month_end"] <= start_cutoff]["ranked_multi_factor_rotation_equity"].iloc[-1] if (df["month_end"] <= start_cutoff).any() else 1.0
    end_equity = window["ranked_multi_factor_rotation_equity"].iloc[-1]
    return {
        "years": years,
        "start": window["month_end"].iloc[0].date(),
        "end": window["month_end"].iloc[-1].date(),
        "total_return": end_equity / start_equity - 1,
        "months": len(window),
    }


sub_stats = pd.DataFrame([sub_window_stats(curve, y) for y in (5, 10, 15)]).set_index("years")
sub_stats["total_return"] = sub_stats["total_return"].map(lambda v: f"{v:+.2%}" if pd.notna(v) else "n/a")
sub_stats
